# Re-route 10,378 Hinglish prompts (fine-tuned Laya, Colab T4)

**Inputs** (Google Drive folder `laya_ft/`): `laya_ckpt_ft.zip`, `prompts.csv` (single `prompt` column).
**Output:** `prompts_routed_v2.csv` — download back when done. Time: **~5-10 min**.


## 1. GPU check


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('cuda:', torch.cuda.is_available(), '| gpus:', torch.cuda.device_count())
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> GPU'


## 2. Install


In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" safetensors pandas
import laya
print('laya:', laya.__version__)


## 3. Mount Drive, unpack fine-tuned checkpoint


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, csv
FT = '/content/drive/MyDrive/laya_ft'
CKPT = '/content/laya_ckpt_ft'
os.makedirs(CKPT, exist_ok=True)
assert os.path.exists(f'{FT}/laya_ckpt_ft.zip'), f'put laya_ckpt_ft.zip in {FT}'
assert os.path.exists(f'{FT}/prompts.csv'), f'put prompts.csv in {FT}'
with zipfile.ZipFile(f'{FT}/laya_ckpt_ft.zip') as z:
    z.extractall(CKPT)
import json
print('ckpt:', json.load(open(f'{CKPT}/rl_agent_config.json')).get('model_name'))


## 4. Route all prompts (batched, flushed per chunk, resumable)


In [ ]:
import time
SRC = f'{FT}/prompts.csv'
DST = '/content/prompts_routed_v2.csv'
QUESTIONS = {'cheap': {'type': 'choice',
    'instructions': ('Can a small cheap language model handle this request correctly '
                     'on its own, or does it need an expensive premium model?'),
    'criteria': {'Yes': 'simple routine request a small model handles well',
               'No': ('needs a premium model: hard reasoning, code, '
                      'expert advice, or high-stakes accuracy')}}}
prompts = [r['prompt'] for r in csv.DictReader(open(SRC, encoding='utf-8-sig'))]
done = 0
if os.path.exists(DST):
    done = sum(1 for _ in open(DST, encoding='utf-8')) - 1
    print(f'resuming at row {done}')
agent = laya.load(CKPT, device='cuda')
print('agent ready', flush=True)
t0 = time.time()
mode = 'a' if done > 0 else 'w'
with open(DST, mode, encoding='utf-8', newline='') as f:
    w = csv.writer(f)
    if done == 0:
        w.writerow(['prompt', 'cheap_ok', 'prob_yes', 'prob_no', 'answer_confidence', 'model'])
    for s in range(done, len(prompts), 160):
        chunk = prompts[s:s + 160]
        res = agent.predict_batch(chunk, QUESTIONS, batch_size=32, sort_by_length=True)
        for p, r in zip(chunk, res):
            a = r['answers']['cheap']
            pr = a.get('probabilities', {})
            w.writerow([p, a['choice'], pr.get('Yes', ''), pr.get('No', ''),
                        a.get('answer_confidence', ''), 'english'])
        f.flush()
        dt = time.time() - t0
        nd = min(s + 160, len(prompts))
        print(f'{nd}/{len(prompts)} ({nd/len(prompts):.1%}) elapsed={dt/60:.1f}m', flush=True)
print('ALL DONE ->', DST)


## 5. Sanity check + download

Check: row count = 10378 + header, Yes/No split printed. Then download
`/content/prompts_routed_v2.csv` (Files pane → right-click → Download) into
your local `backend/experiments/cascade_routing/data/`.


In [ ]:
import csv
from collections import Counter
rows = list(csv.DictReader(open('/content/prompts_routed_v2.csv', encoding='utf-8')))
print('rows:', len(rows))
print(Counter(r['cheap_ok'] for r in rows))
